# 🎯 Phase 2: Milestone Exam Solutions

> **Functions, Modularity & Data Wrangling**
>
> This notebook contains comprehensive solutions for all four Phase Milestone Exam questions.
> Each solution demonstrates the synthesis of concepts from Days 13-24.

---

## Question 1: Data Pipeline

**Combines**: File Handling (Day 16), Regex (Day 17), Pandas (Day 23), Exception Handling (Day 15)

**Scenario**: Build a data pipeline that:
1. Reads multiple CSV files from a directory
2. Validates email format using regex
3. Cleans and standardizes names
4. Merges into a single DataFrame
5. Handles missing files gracefully

In [ ]:
import re
import pandas as pd
from io import StringIO

# --- Simulated CSV files (inline data for Pyodide compatibility) ---
csv_data_files = {
    "customers_jan.csv": """name,email,phone,city
  JOHN DOE,john.doe@email.com,555-123-4567,New York
jane smith,JANE@COMPANY.ORG,555.987.6543,Chicago
Bob Wilson,bob@test.com,555-123-4567,Los Angeles""",
    "customers_feb.csv": """name,email,phone,city
  alice jones,alice@email.com,invalid-phone,Seattle
john doe,johndoe@email.com,555-111-2222,New York
MARY JOHNSON,mary.j@test,555-333-4444,Miami""",
    "customers_mar.csv": """name,email,phone,city
carlos garcia,carlos@corp.com,555-555-6666,Houston
  Emily Zhang,emily@valid.org,555-777-8888,Denver
FRANK MILLER,frank@bad email.com,555-999-0000,Portland"""
}

print(f"Loaded {len(csv_data_files)} simulated CSV files")

In [ ]:
def validate_email(email):
    """
    Validate an email address using regex.

    Args:
        email: Email string to validate

    Returns:
        bool: True if valid email format
    """
    if not isinstance(email, str) or not email.strip():
        return False
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, email.strip()))


def clean_name(name):
    """
    Clean and standardize a name string.

    Args:
        name: Raw name string

    Returns:
        str: Title-cased, trimmed name
    """
    if not isinstance(name, str):
        return ""
    return name.strip().title()


def clean_phone(phone):
    """
    Extract and validate phone number (10 digits).

    Args:
        phone: Raw phone string

    Returns:
        str or None: Cleaned 10-digit phone, or None if invalid
    """
    if not isinstance(phone, str):
        return None
    digits = re.sub(r'\D', '', phone)
    return digits if len(digits) == 10 else None

In [ ]:
def process_data_files(file_dict):
    """
    Process all CSV data sources into a single clean DataFrame.

    In a real environment, this would use pathlib.Path.glob("*.csv")
    to discover files on disk. Here we use in-memory StringIO for
    Pyodide browser compatibility.

    Args:
        file_dict: Dictionary mapping filename to CSV string content

    Returns:
        pd.DataFrame: Cleaned, merged DataFrame
    """
    frames = []
    errors = []

    for filename, content in file_dict.items():
        try:
            df = pd.read_csv(StringIO(content))
            df["source_file"] = filename
            frames.append(df)
            print(f"  ✅ Loaded {filename}: {len(df)} rows")
        except Exception as e:
            errors.append((filename, str(e)))
            print(f"  ❌ Failed {filename}: {e}")

    if not frames:
        raise ValueError("No valid data files found")

    # Merge all DataFrames
    combined = pd.concat(frames, ignore_index=True)

    # Clean names
    combined["name"] = combined["name"].apply(clean_name)

    # Validate emails
    combined["email_valid"] = combined["email"].apply(validate_email)
    combined["email"] = combined["email"].apply(
        lambda x: x.strip().lower() if isinstance(x, str) else x
    )

    # Clean phones
    combined["phone"] = combined["phone"].apply(clean_phone)

    # Standardize city names
    combined["city"] = combined["city"].apply(
        lambda x: x.strip().title() if isinstance(x, str) else x
    )

    if errors:
        print(f"\n⚠️  {len(errors)} file(s) had errors")

    return combined

In [ ]:
# Test the Data Pipeline
print("=" * 50)
print("DATA PIPELINE TEST")
print("=" * 50)

# Add a "bad" file to test error handling
test_files = {**csv_data_files, "missing.csv": None}
# Simulate a bad file by setting content to something unparseable
test_files["corrupt.csv"] = "this,is\nnot,valid,csv,data,extra"

print("\nProcessing files:")
result = process_data_files(test_files)

print(f"\n{'=' * 50}")
print(f"Total records: {len(result)}")
print(f"Valid emails: {result['email_valid'].sum()}")
print(f"Valid phones: {result['phone'].notna().sum()}")
print(f"\nCleaned Data:")
print(result[["name", "email", "email_valid", "phone", "city"]].to_string(index=False))

---

## Question 2: Sales Analytics Class

**Combines**: OOP (Day 18), Pandas (Day 23-24), DateTime (Day 19), GroupBy

**Scenario**: Create a `SalesAnalyzer` class that loads sales data, calculates YoY growth, identifies top products, and generates summary statistics.

In [ ]:
import pandas as pd
from io import StringIO

# Synthetic sales data (inline for Pyodide compatibility)
SALES_CSV = """date,product,category,quantity,price,region
2023-01-15,Widget A,Electronics,10,29.99,North
2023-02-10,Gadget X,Home,20,14.99,North
2023-03-05,Tool Z,Hardware,15,9.99,East
2023-04-01,Widget B,Electronics,12,49.99,South
2023-05-15,Widget A,Electronics,6,29.99,East
2023-06-10,Gadget X,Home,25,14.99,West
2023-07-22,Widget B,Electronics,18,49.99,North
2023-08-01,Tool Z,Hardware,30,9.99,North
2023-09-15,Widget A,Electronics,14,29.99,South
2023-10-05,Gadget X,Home,22,14.99,East
2023-11-18,Widget B,Electronics,20,49.99,West
2023-12-01,Tool Z,Hardware,10,9.99,North
2024-01-10,Widget A,Electronics,15,31.99,North
2024-02-14,Gadget X,Home,28,15.99,North
2024-03-10,Widget B,Electronics,22,52.99,East
2024-04-05,Tool Z,Hardware,18,10.99,West
2024-05-20,Widget A,Electronics,12,31.99,South
2024-06-15,Gadget X,Home,30,15.99,North
2024-07-01,Widget B,Electronics,25,52.99,North
2024-08-12,Tool Z,Hardware,20,10.99,East
2024-09-05,Widget A,Electronics,18,31.99,West
2024-10-22,Gadget X,Home,35,15.99,South
2024-11-15,Widget B,Electronics,28,52.99,North
2024-12-01,Tool Z,Hardware,22,10.99,North"""

print("Sales data loaded")

In [ ]:
class SalesAnalyzer:
    """
    Analyze sales data with year-over-year growth, top products,
    and time-period summaries.

    Attributes:
        df: The loaded and parsed sales DataFrame
    """

    def __init__(self, csv_string):
        """
        Load and prepare sales data.

        Args:
            csv_string: CSV content as a string (or file path in production)
        """
        self.df = pd.read_csv(StringIO(csv_string), parse_dates=["date"])
        self.df["revenue"] = self.df["quantity"] * self.df["price"]
        self.df["year"] = self.df["date"].dt.year
        self.df["month"] = self.df["date"].dt.month

    def yoy_growth(self, column="revenue"):
        """
        Calculate year-over-year growth for a given metric.

        Args:
            column: Column to aggregate (default: 'revenue')

        Returns:
            pd.DataFrame: Year, total, and growth percentage
        """
        yearly = self.df.groupby("year")[column].sum().reset_index()
        yearly["prev_year"] = yearly[column].shift(1)
        yearly["growth_pct"] = (
            (yearly[column] - yearly["prev_year"]) / yearly["prev_year"] * 100
        ).round(2)
        return yearly

    def top_products(self, n=5):
        """
        Identify top N products by total revenue.

        Args:
            n: Number of products to return (default: 5)

        Returns:
            pd.DataFrame: Product, total revenue, and quantity sold
        """
        product_stats = (
            self.df.groupby("product")
            .agg(
                total_revenue=("revenue", "sum"),
                total_quantity=("quantity", "sum"),
                avg_price=("price", "mean"),
            )
            .sort_values("total_revenue", ascending=False)
            .head(n)
        )
        return product_stats.round(2)

    def summary_by_period(self, period="M"):
        """
        Generate summary statistics by time period.

        Args:
            period: Pandas offset alias - 'M' for monthly, 'Q' for quarterly

        Returns:
            pd.DataFrame: Period-level summary
        """
        resampled = (
            self.df.set_index("date")
            .resample(period)
            .agg(
                total_revenue=("revenue", "sum"),
                orders=("quantity", "count"),
                avg_order_value=("revenue", "mean"),
            )
        )
        return resampled.round(2)

In [ ]:
# Test the Sales Analytics Class
print("=" * 50)
print("SALES ANALYTICS CLASS TEST")
print("=" * 50)

analyzer = SalesAnalyzer(SALES_CSV)

# Year-over-Year Growth
print("\n📈 Year-over-Year Revenue Growth:")
yoy = analyzer.yoy_growth()
for _, row in yoy.iterrows():
    growth_str = f"{row['growth_pct']:+.1f}%" if pd.notna(row['growth_pct']) else "(baseline)"
    print(f"  {int(row['year'])}: ${row['revenue']:,.2f} {growth_str}")

# Top Products
print("\n🏆 Top Products by Revenue:")
top = analyzer.top_products(n=4)
for product, row in top.iterrows():
    print(f"  {product}: ${row['total_revenue']:,.2f} ({int(row['total_quantity'])} units)")

# Quarterly Summary
print("\n📊 Quarterly Summary:")
quarterly = analyzer.summary_by_period("Q")
print(quarterly.to_string())

---

## Question 3: Log Analyzer

**Combines**: Regex (Day 17), File Handling (Day 16), Higher-Order Functions (Day 13)

**Scenario**: Parse log files to extract timestamps, log levels, error messages, aggregate errors by type, and find time patterns.

In [ ]:
import re
from collections import Counter, defaultdict
from datetime import datetime

# Inline log data (in production, you'd read from a file)
LOG_DATA = """[2024-01-15 08:30:45] INFO: Application started successfully
[2024-01-15 08:31:02] INFO: Database connection established
[2024-01-15 09:15:33] WARNING: Slow query detected (2.3s)
[2024-01-15 09:45:12] ERROR: Database connection failed - timeout after 30s
[2024-01-15 10:00:01] INFO: Scheduled backup initiated
[2024-01-15 10:15:45] ERROR: File not found: /data/reports/q4.csv
[2024-01-15 11:30:22] INFO: User login: admin@company.com
[2024-01-15 12:00:00] WARNING: Memory usage at 85%
[2024-01-15 13:45:18] ERROR: Database connection failed - connection refused
[2024-01-15 14:00:33] INFO: Cache cleared successfully
[2024-01-15 14:30:45] ERROR: Authentication failed for user: guest
[2024-01-15 15:15:00] WARNING: Disk space below 10%
[2024-01-15 16:00:12] ERROR: Database connection failed - SSL handshake error
[2024-01-15 16:45:30] INFO: Daily report generated
[2024-01-15 17:00:00] INFO: Application shutdown gracefully
[2024-01-16 08:30:00] INFO: Application started successfully
[2024-01-16 08:31:15] ERROR: Configuration file missing: config.yaml
[2024-01-16 09:00:22] WARNING: API rate limit approaching (90%)
[2024-01-16 10:30:45] ERROR: File not found: /data/exports/users.json
[2024-01-16 11:15:33] INFO: Batch processing completed: 1500 records
[2024-01-16 12:00:00] ERROR: Database connection failed - too many connections
[2024-01-16 13:30:18] WARNING: Slow query detected (5.1s)
[2024-01-16 14:45:00] ERROR: Authentication failed for user: readonly
[2024-01-16 15:00:30] INFO: System health check passed
[2024-01-16 16:30:12] ERROR: Database connection failed - timeout after 30s"""

print(f"Log data loaded: {len(LOG_DATA.strip().splitlines())} lines")

In [ ]:
def parse_log_line(line):
    """
    Parse a single log line into structured data.

    Log format: [YYYY-MM-DD HH:MM:SS] LEVEL: Message

    Args:
        line: Raw log line string

    Returns:
        dict or None: Parsed log entry with timestamp, level, message
    """
    pattern = r'\[(.+?)\] (\w+): (.+)'
    match = re.match(pattern, line.strip())

    if not match:
        return None

    timestamp_str, level, message = match.groups()
    timestamp = datetime.strptime(timestamp_str, "%Y-%m-%d %H:%M:%S")

    return {
        "timestamp": timestamp,
        "level": level,
        "message": message,
        "hour": timestamp.hour,
        "date": timestamp.date(),
    }


def categorize_error(message):
    """
    Categorize an error message into a general type.

    Args:
        message: Error message string

    Returns:
        str: Error category
    """
    categories = {
        r'[Dd]atabase|[Cc]onnection': 'Database',
        r'[Ff]ile not found': 'FileSystem',
        r'[Aa]uthentication|[Ll]ogin': 'Authentication',
        r'[Cc]onfiguration|[Cc]onfig': 'Configuration',
        r'[Mm]emory|[Dd]isk': 'Resources',
    }
    for pattern, category in categories.items():
        if re.search(pattern, message):
            return category
    return 'Other'

In [ ]:
def analyze_logs(log_text):
    """
    Perform comprehensive analysis of log data.

    Args:
        log_text: Multi-line string of log entries

    Returns:
        dict: Structured analysis with counts, categories, and patterns
    """
    lines = log_text.strip().splitlines()

    # Parse all lines (filter out unparseable)
    entries = list(filter(None, map(parse_log_line, lines)))

    # Count by level
    level_counts = Counter(e["level"] for e in entries)

    # Categorize errors
    errors = [e for e in entries if e["level"] == "ERROR"]
    error_categories = Counter(categorize_error(e["message"]) for e in errors)

    # Errors per hour
    errors_by_hour = Counter(e["hour"] for e in errors)

    # Errors per day
    errors_by_date = Counter(str(e["date"]) for e in errors)

    # Peak error hour
    peak_hour = errors_by_hour.most_common(1)[0] if errors_by_hour else (None, 0)

    return {
        "total_lines": len(lines),
        "parsed_entries": len(entries),
        "level_counts": dict(level_counts),
        "error_categories": dict(error_categories),
        "errors_by_hour": dict(sorted(errors_by_hour.items())),
        "errors_by_date": dict(errors_by_date),
        "peak_error_hour": peak_hour,
        "error_messages": [e["message"] for e in errors],
    }

In [ ]:
# Test the Log Analyzer
print("=" * 50)
print("LOG ANALYZER TEST")
print("=" * 50)

analysis = analyze_logs(LOG_DATA)

print(f"\nTotal log lines: {analysis['total_lines']}")
print(f"Successfully parsed: {analysis['parsed_entries']}")

print("\n📊 Log Levels:")
for level, count in sorted(analysis['level_counts'].items()):
    bar = '█' * count
    print(f"  {level:10s} {count:3d} {bar}")

print("\n🔴 Error Categories:")
for category, count in sorted(analysis['error_categories'].items(), key=lambda x: -x[1]):
    print(f"  {category:20s} {count}")

print("\n⏰ Errors by Hour:")
for hour, count in analysis['errors_by_hour'].items():
    bar = '█' * count
    print(f"  {hour:02d}:00  {count}  {bar}")

peak_h, peak_c = analysis['peak_error_hour']
print(f"\n🔥 Peak error hour: {peak_h:02d}:00 ({peak_c} errors)")

print("\n📋 Error Messages:")
for msg in analysis['error_messages']:
    print(f"  → {msg}")

---

## Question 4: Virtual Environment Automation

**Combines**: Modules (Day 14), File Handling (Day 16), Exception Handling (Day 15)

**Scenario**: Create a project scaffolding tool that creates a directory structure, generates a `requirements.txt`, `README.md`, and `.gitignore`.

> **⚠️ Pyodide Note**: `subprocess` and `venv` are not available in the browser runtime.
> This solution demonstrates the scaffolding logic using `pathlib` for directory/file creation.
> In a real environment, you'd also call `subprocess.run([sys.executable, '-m', 'venv', 'venv'])`.

In [ ]:
from pathlib import Path
import tempfile


# --- Template Constants ---
GITIGNORE_TEMPLATE = """# Python
__pycache__/
*.py[cod]
*$py.class
*.egg-info/
dist/
build/

# Virtual Environments
venv/
.venv/
env/

# IDE
.idea/
.vscode/
*.swp

# OS
.DS_Store
Thumbs.db
"""

README_TEMPLATE = """# {name}

{description}

## Setup

```bash
python -m venv venv
source venv/bin/activate  # On Windows: venv\\Scripts\\activate
pip install -r requirements.txt
```

## Usage

```bash
python src/main.py
```

## Testing

```bash
python -m pytest tests/
```
"""

MAIN_TEMPLATE = """\"\"\"Main entry point for {name}.\"\"\"\n\n\ndef main():\n    print(\"Hello from {name}!\")\n\n\nif __name__ == \"__main__\":\n    main()\n"""

In [ ]:
def create_project(name, description="A Python project", packages=None, base_dir=None):
    """
    Create a new Python project with standard structure.

    Creates:
        project_name/
        ├── src/
        │   ├── __init__.py
        │   └── main.py
        ├── tests/
        │   └── __init__.py
        ├── requirements.txt
        ├── README.md
        └── .gitignore

    Args:
        name: Project name (used for directory and README)
        description: Project description for README
        packages: List of pip packages for requirements.txt
        base_dir: Parent directory (default: current directory)

    Returns:
        Path: The created project directory path
    """
    if packages is None:
        packages = []

    if base_dir is None:
        base_dir = Path.cwd()
    else:
        base_dir = Path(base_dir)

    project_dir = base_dir / name

    # Create directories
    dirs_to_create = [
        project_dir / "src",
        project_dir / "tests",
    ]

    for d in dirs_to_create:
        d.mkdir(parents=True, exist_ok=True)
        print(f"  📁 Created: {d.relative_to(base_dir)}")

    # Create files
    files_to_create = {
        project_dir / ".gitignore": GITIGNORE_TEMPLATE,
        project_dir / "README.md": README_TEMPLATE.format(name=name, description=description),
        project_dir / "requirements.txt": "\n".join(packages) + "\n" if packages else "# Add dependencies here\n",
        project_dir / "src" / "__init__.py": "",
        project_dir / "src" / "main.py": MAIN_TEMPLATE.format(name=name),
        project_dir / "tests" / "__init__.py": "",
    }

    for filepath, content in files_to_create.items():
        filepath.write_text(content)
        print(f"  📄 Created: {filepath.relative_to(base_dir)}")

    # In production, you would also create a virtual environment:
    # import subprocess, sys
    # subprocess.run([sys.executable, '-m', 'venv', str(project_dir / 'venv')])
    print(f"\n  ⚠️  Skipping venv creation (not supported in browser runtime)")
    print(f"  💡 In production, run: python -m venv {name}/venv")

    return project_dir

In [ ]:
# Test the Project Scaffolding Tool
print("=" * 50)
print("PROJECT SCAFFOLDING TEST")
print("=" * 50)

# Use a temp directory so we don't pollute the filesystem
with tempfile.TemporaryDirectory() as tmpdir:
    print(f"\nCreating project in: {tmpdir}\n")

    project_path = create_project(
        name="my_data_app",
        description="A data analysis application for sales reporting.",
        packages=["pandas>=2.0", "numpy>=1.24", "matplotlib>=3.7", "pytest>=7.0"],
        base_dir=tmpdir,
    )

    # Verify the structure
    print(f"\n{'=' * 50}")
    print("Generated Project Structure:")
    print("=" * 50)

    for item in sorted(project_path.rglob("*")):
        depth = len(item.relative_to(project_path).parts)
        indent = "  " * depth
        icon = "📁" if item.is_dir() else "📄"
        print(f"{indent}{icon} {item.name}")

    # Show generated README
    print(f"\n{'=' * 50}")
    print("Generated README.md:")
    print("=" * 50)
    print((project_path / "README.md").read_text())

    # Show generated requirements.txt
    print("Generated requirements.txt:")
    print("-" * 30)
    print((project_path / "requirements.txt").read_text())

---

## 🎓 Summary

This notebook demonstrated solutions to all four Phase 2 Milestone Exam questions:

1. **Data Pipeline**: Combined file handling, regex validation, pandas merging, and exception handling
2. **Sales Analytics Class**: Built an OOP class with YoY growth, top products, and period summaries
3. **Log Analyzer**: Used regex parsing, higher-order functions (`map`, `filter`), and `Counter` for aggregation
4. **Virtual Environment Automation**: Demonstrated project scaffolding with `pathlib` templates

Each solution follows best practices including:
- Clear docstrings with Args/Returns documentation
- Graceful error handling for edge cases
- Modular, reusable function design
- Pyodide-compatible implementations (inline data, no filesystem dependencies)